# 1. Run PyNNLF: ASHD 148 Households

Runs missing ASHD `ds20` experiments for 30-minute, 1-day, and 1-week horizons. The run cell is resumable.

## 1. Setup And Batch Spec

In [ ]:
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")
PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
import pandas as pd
import yaml
sys.path.insert(0, str(REPO_ROOT / "src"))
import pynnlf  # noqa: E402
BATCH_PATH = PROJECT_DIR / "specs" / "ashd_148hh_batch.yaml"
RESULTS_ROOT = PROJECT_DIR / "experiment_result"
TEMP_SPEC_PATH = PROJECT_DIR / "specs" / "_tmp_ashd_148hh_single.yaml"
batch = yaml.safe_load(BATCH_PATH.read_text(encoding="utf-8"))
config = yaml.safe_load((PROJECT_DIR / "specs" / "pynnlf_config.yaml").read_text(encoding="utf-8"))
forecast_horizon_minutes = {key: int(value) for key, value in config["forecast_horizons"].items()}
print(batch)

## 2. Validate Input Dataset

In [ ]:
DATASET_FILE = PROJECT_DIR / "data" / "ds20_ashd_148hh_with_weather.csv"
EXPECTED_COLUMNS = ["datetime", "netload_kW", "air_temperature_in_degrees_c", "relative_humidity_in_percentage", "wind_speed_in_km_h"]
if not DATASET_FILE.exists():
    raise FileNotFoundError(DATASET_FILE)
df = pd.read_csv(DATASET_FILE, parse_dates=["datetime"])
if list(df.columns) != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected columns in {DATASET_FILE.name}: {list(df.columns)}")
if len(df) != 52608:
    raise ValueError(f"Expected 52,608 rows, found {len(df):,}")
if df["datetime"].duplicated().any() or df.isna().any().any():
    raise ValueError("Dataset has duplicate timestamps or missing values")
if df["datetime"].diff().dropna().nunique() != 1 or df["datetime"].diff().dropna().iloc[0] != pd.Timedelta(minutes=30):
    raise ValueError("Dataset is not regular 30-minute data")
print(f"{DATASET_FILE.name}: OK | {df['datetime'].min()} to {df['datetime'].max()}")
display(df.head())

## 3. Run Missing Experiments

This is the only cell that runs PyNNLF. It writes one temporary single-run spec at a time.

In [ ]:
def completed_keys(results_root):
    keys = set()
    for result_file in sorted(results_root.glob("E*/E*_a1_experiment_result.csv")):
        try:
            row = pd.read_csv(result_file, nrows=1).iloc[0]
            keys.add((str(row.get("dataset_no", "")), int(row.get("forecast_horizon_min")), str(row.get("model_no", "")), str(row.get("hyperparameter_no", ""))))
        except Exception:
            continue
    return keys
done = completed_keys(RESULTS_ROOT)
total = len(batch["datasets"]) * len(batch["forecast_horizons"]) * len(batch["model_and_hp"])
run_index = 0
for dataset_id in batch["datasets"]:
    for forecast_horizon_id in batch["forecast_horizons"]:
        horizon_minutes = forecast_horizon_minutes[forecast_horizon_id]
        for model_id, hp in batch["model_and_hp"]:
            run_index += 1
            key = (str(dataset_id), horizon_minutes, str(model_id), str(hp))
            if key in done:
                print(f"[skip {run_index}/{total}] {dataset_id} {forecast_horizon_id} {model_id} {hp}")
                continue
            print(f"[run {run_index}/{total}] {dataset_id} {forecast_horizon_id} {model_id} {hp}")
            temp_spec = {"datasets": [dataset_id], "forecast_horizons": [forecast_horizon_id], "model_and_hp": [[model_id, hp]]}
            TEMP_SPEC_PATH.write_text(yaml.safe_dump(temp_spec, sort_keys=False), encoding="utf-8")
            pynnlf.run_experiment_batch(TEMP_SPEC_PATH, plot_enabled=True)
            done.add(key)
if TEMP_SPEC_PATH.exists():
    TEMP_SPEC_PATH.unlink()
pynnlf.recap_experiments(RESULTS_ROOT)
print(f"Recap written: {RESULTS_ROOT / 'a1_experiment_result.csv'}")

## 4. Recap Check

In [ ]:
recap = pd.read_csv(RESULTS_ROOT / "a1_experiment_result.csv")
ashd_recap = recap.loc[recap["dataset_no"].eq("ds20")].copy().sort_values(["forecast_horizon_min", "model_name"])
print(f"ASHD ds20 recap rows: {len(ashd_recap):,}")
display(ashd_recap[["dataset_no", "forecast_horizon_min", "model_name", "test_nRMSE", "test_nRMSE_stddev"]])